
# การติดตามพายุหมุนเขตร้อน

การติดตามพายุหมุนเขตร้อนด้วยโมเดลการวินิจฉัยตัวติดตาม

ตัวอย่างนี้จะสาธิตวิธีใช้การวินิจฉัยตัวติดตามพายุหมุนเขตร้อน (TC)
โมเดลสำหรับการสร้างเส้นทาง TC
การวินิจฉัยที่ใช้ในที่นี้สามารถใช้ร่วมกับโมเดลสภาพอากาศ AI อื่นๆ และ ensemble ได้
วิธีการสร้าง inference workflow ที่ซับซ้อนซึ่งเปิดใช้งานการวิเคราะห์ดาวน์สตรีม


ในตัวอย่างนี้คุณจะได้เรียนรู้:

- วิธีสร้างอินสแตนซ์การวินิจฉัยตัวติดตาม TC
- วิธีใช้ TC tracker กับข้อมูล
- วิธีจับคู่ TC tracker กับ Prognostic Model
- วิธีทำ post-processing กับผลลัพธ์


In [ ]:
# /// script
# dependencies = [
#   "torch==2.9.1", # Match lock file to avoid torch-harmonics issue
#   "earth2studio[cyclone,sfno] @ git+https://github.com/NVIDIA/earth2studio.git",
#   "cartopy",
# ]
# ///

## การเตรียมองค์ประกอบ
ตัวอย่างนี้จะพิจารณาการติดตามพายุไซโคลนในช่วงเดือนสิงหาคม พ.ศ. 2552 ซึ่งเป็นช่วงเวลาหนึ่ง
พายุหมุนเขตร้อนหลายลูกที่ส่งผลกระทบต่อเอเชียตะวันออก
Earth2Studio มีตัวติดตาม TC หลากหลายรูปแบบ เช่น :py:class:`earth2studio.models.dx.TCTrackerVitart`
และ :py:class:`earth2studio.models.dx.TCTrackerWuDuan`
ความแตกต่างคืออัลกอริธึมพื้นฐานที่ใช้ในการระบุศูนย์กลาง

ตัวอย่างนี้ต้องใช้องค์ประกอบต่อไปนี้:

- โมเดลการวินิจฉัย: ใช้ตัวติดตาม TC :py:class:`earth2studio.models.dx.TCTrackerWuDuan`
- Datasource: ดึงข้อมูลจาก WB2 ERA5 data API ผ่าน :py:class:`earth2studio.data.WB2ERA5`.
- Prognostic Model: ใช้ FourCastNet รุ่น :py:class:`earth2studio.models.px.FCN` ในตัว



In [ ]:
import os

os.makedirs("outputs", exist_ok=True)
from dotenv import load_dotenv

load_dotenv()  # สิ่งที่ต้องทำ: สร้างฟังก์ชันการเตรียมตัวอย่างทั่วไป

from datetime import datetime, timedelta

import torch

from earth2studio.data import ARCO
from earth2studio.models.dx import TCTrackerWuDuan
from earth2studio.models.px import SFNO
from earth2studio.utils.time import to_time_array

# สร้างตัวติดตามพายุหมุนเขตร้อน
tracker = TCTrackerWuDuan()

# โหลดmodel packageเริ่มต้นซึ่งดาวน์โหลดcheckpointจาก NGC
package = SFNO.load_default_package()
prognostic = SFNO.load_model(package)

# สร้างแหล่งข้อมูล
data = ARCO()

nsteps = 16  # จำนวนขั้นตอนในการเรียกใช้ตัวติดตามในอนาคต
start_time = datetime(2009, 8, 5)  # วันที่เริ่มต้นสำหรับ inference

## ข้อมูลการวิเคราะห์การติดตาม
ก่อนที่จะเชื่อมต่อตัวติดตาม TC กับ Prognostic Model เราจะนำไปใช้กับก่อน
ข้อมูลการวิเคราะห์
เราสามารถดึงช่วงเวลาเล็กๆ จากแหล่งข้อมูลและมอบให้กับโมเดลของเรา

สำหรับการพยากรณ์ เราจะคาดการณ์เป็นเวลาสองวัน (ซึ่งจะถูกดำเนินการเป็น batch) สำหรับ
20 forecast steps ซึ่งก็คือ 5 วัน



In [ ]:
from earth2studio.data import fetch_data, prep_data_array

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
tracker = tracker.to(device)

# แผ่นดินถล่มเกิดขึ้นเมื่อวันที่ 25 สิงหาคม 2560
times = [start_time + timedelta(hours=6 * i) for i in range(nsteps + 1)]
for step, time in enumerate(times):
    da = data(time, tracker.input_coords()["variable"])
    x, coords = prep_data_array(da, device=device)
    output, output_coords = tracker(x, coords)
    print(f"Step {step}: ARCO tracks output shape {output.shape}")

era5_tracks = output.cpu()
torch.save(era5_tracks, "outputs/13_era5_paths.pt")

โปรดสังเกตว่าเอาท์พุตเทนเซอร์จะเพิ่มขึ้นเมื่อมีการวนซ้ำ
เนื่องจากตัวติดตามสร้างแทร็กตามการส่งต่อครั้งก่อนหน้าที่ส่งคืน
เทนเซอร์ที่มีขนาด [batch, เส้นทาง, ขั้นตอน, ตัวแปร]
เส้นทางทั้งหมดไม่ได้รับประกันว่าจะมีความยาวเท่ากันหรือมีเวลาเริ่มต้น/หยุดเท่ากัน
ดังนั้นข้อมูลที่ขาดหายไปจะถูกเติมด้วยค่าน่าน

ถัดไป เรามาทำซ้ำขั้นตอนเดียวกันโดยใช้โมเดล AI เชิงพยากรณ์
เราสามารถใช้หนึ่งในบิลด์ใน workflows ได้ แต่ที่นี่เราจะใช้งาน
inference ลูป



In [ ]:
from tqdm import tqdm

from earth2studio.utils.coords import map_coords

prognostic = prognostic.to(device)
# รีเซ็ตบัฟเฟอร์เส้นทางภายในของตัวติดตาม
tracker.reset_path_buffer()

# โหลดสถานะเริ่มต้น
x, coords = fetch_data(
    source=data,
    time=to_time_array([start_time]),
    variable=prognostic.input_coords()["variable"],
    lead_time=prognostic.input_coords()["lead_time"],
    device=device,
)

# สร้างตัววนซ้ำเชิงพยากรณ์
model = prognostic.create_iterator(x, coords)
with tqdm(total=nsteps + 1, desc="Running inference") as pbar:
    for step, (x, coords) in enumerate(model):
        # เรียกใช้ตัวติดตาม
        x, coords = map_coords(x, coords, tracker.input_coords())
        output, output_coords = tracker(x, coords)
        # มาลบหรี่ lead time กัน
        output = output[:, 0]
        print(f"Step {step}: SFNO tracks output shape {output.shape}")

        pbar.update(1)
        if step == nsteps:
            break

sfno_tracks = output.cpu()
torch.save(sfno_tracks, "outputs/13_sfno_paths.pt")

โปรดทราบว่าก่อนการวนซ้ำ inference ของโมเดล AI จะเป็นบัฟเฟอร์พาธของตัวติดตาม
ถูกรีเซ็ตซึ่งจะรีเฟรชสถานะเส้นทางของตัวติดตามโดยเริ่มจากศูนย์
มิฉะนั้นจะพยายามผนวกเข้ากับแทร็กที่มีอยู่จากแหล่งข้อมูล
วนซ้ำในส่วนก่อนหน้า

ในที่สุดเราก็สามารถวางแผนผลลัพธ์เพื่อเปรียบเทียบความจริงของสนามแข่งจาก ERA5 ได้
ที่ผลิตโดย SFNO
เรียกคืนเอาต์พุตของ :py:class:`earth2studio.models.dx.TCTrackerWuDuan` ที่มีเส้นทาง
ID ในมิติที่สอง นั่นคือสิ่งที่จะเป็นตัวกำหนดจำนวนบรรทัด
พิกัดละติจูด/ลอนเป็นตัวแปรสองตัวแรกในมิติสุดท้าย
สุดท้ายนี้ เราแค่ต้องคำนึงถึงค่าตัวเติม NaN ที่สามารถหาได้ง่าย
ปิดบังและ "เส้นทาง" ใด ๆ ที่มีความยาวไม่เกิน 2 จุด



In [ ]:
from datetime import datetime, timedelta

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

# แปลงแทร็กจากเทนเซอร์เป็นอาร์เรย์จำนวนมาก
era5_paths = era5_tracks.numpy()
sfno_paths = sfno_tracks.numpy()

# คำนวณวันที่สิ้นสุด
end_time = start_time + timedelta(hours=6 * nsteps)

# สร้างฟิกเกอร์ด้วย cartopy projection
plt.figure(figsize=(10, 8))
projection = ccrs.LambertConformal(
    central_longitude=130.0, central_latitude=30.0, standard_parallels=(20.0, 40.0)
)
ax = plt.axes(projection=projection)

# เพิ่มคุณสมบัติแผนที่
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.LAND, alpha=0.1)
ax.gridlines(draw_labels=True, alpha=0.6)
ax.set_extent([90, 170, 0, 50], crs=ccrs.PlateCarree())

era5_cmap = plt.cm.autumn
sfno_cmap = plt.cm.winter

for path in range(era5_paths.shape[1]):
    # รับพิกัดละติจูด/ลอน โดยกรองนาโนออก
    lats = era5_paths[0, path, :, 0]
    lons = era5_paths[0, path, :, 1]
    mask = ~np.isnan(lats) & ~np.isnan(lons)
    if mask.any() and len(lons[mask]) > 2:
        color = era5_cmap(path / era5_paths.shape[1])
        ax.plot(
            lons[mask],
            lats[mask],
            color=color,
            linestyle="-.",
            marker="x",
            label="ERA5" if path == 0 else "",
            transform=ccrs.PlateCarree(),
        )

for path in range(sfno_paths.shape[1]):
    # รับพิกัดละติจูด/ลอน โดยกรองนาโนออก
    lats = sfno_paths[0, path, :, 0]
    lons = sfno_paths[0, path, :, 1]
    mask = ~np.isnan(lats) & ~np.isnan(lons)
    if mask.any() and len(lons[mask]) > 2:
        color = sfno_cmap(path / sfno_paths.shape[1])
        ax.plot(
            lons[mask],
            lats[mask],
            color=color,
            linestyle="-",
            label="SFNO" if path == 0 else "",
            transform=ccrs.PlateCarree(),
        )

era5_patch = mpatches.Rectangle(
    (0, 0), 1, 1, fc=era5_cmap(0.3), alpha=0.9, label="ERA5"
)
sfno_patch = mpatches.Rectangle(
    (0, 0), 1, 1, fc=sfno_cmap(0.3), alpha=0.9, label="SFNO"
)
ax.legend(handles=[era5_patch, sfno_patch], loc="upper right", title="Cyclone Tracks")

plt.title(
    f'Tropical Cyclone Tracks\n{start_time.strftime("%Y-%m-%d")} to {end_time.strftime("%Y-%m-%d")}'
)
plt.savefig(f"outputs/13_{start_time}_cyclone_tracks.jpg", bbox_inches="tight", dpi=300)

นอกเหนือจากการกรองค่า NaN แล้ว ผู้ใช้อาจต้องการใช้โพสต์อื่น
ขั้นตอนการประมวลผลบนเส้นทางที่อาจบังคับใช้ความยาวเส้นทางนั้นสูงกว่าที่กำหนด
เกณฑ์หรือตัวกรองตามภูมิศาสตร์อื่นๆ

ไม่มีตัวติดตามพายุไซโคลนที่สมบูรณ์แบบ เราขอแนะนำให้ผู้ใช้ทดลองและปรับแต่งตัวติดตาม
ตามความจำเป็น

